# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [102]:
# 1. Системные и встроенные модули Python
import os
import time
from typing import Tuple

# 2. Библиотеки работы с данными и визуализации
import joblib
import pandas as pd
from IPython.display import Markdown, display
from tqdm.notebook import tqdm

# 3. Метрики и базовая архитектура Scikit-Learn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# 4. Алгоритмы машинного обучения (Классификаторы)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.

**Вход:** Принимает исходный датафрейм, полученный из файла checker_submits.csv, содержащий столбцы uid, labname, numTrials, timestamp.  
**Логика работы:** Извлекает час (hour) из столбца timestamp.Извлекает номер дня недели (weekday) из столбца timestamp (в виде чисел).Удаляет исходный столбец timestamp.  
**Выход:** Возвращает новый измененный датафрейм.

In [103]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Переменная X — это локальное имя для датафрейма, который мы подадим на вход.
        # Проверяем, что к нам пришла именно таблица Pandas
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"Ожидался pd.DataFrame, получен {type(X)}")

        # Защищаем оригинал, создавая копию ячеек
        X_copy = X.copy()

        # Выполняем векторные операции над столбцами
        timestamps = pd.to_datetime(X_copy["timestamp"])
        X_copy["hour"] = timestamps.dt.hour
        X_copy["dayofweek"] = timestamps.dt.weekday

        # Возвращаем измененную матрицу признаков наружу
        return X_copy.drop(columns=["timestamp"])

2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.

**Вход:** Принимает датафрейм, полученный на выходе из предыдущего шага трансформации (FeatureExtractor), и строковое имя целевого столбца (target_column).  
**Логика работы:** Автоматически идентифицирует все категориальные признаки в полученном датафрейме и трансформирует их с помощью стандартного класса OneHotEncoder().  
**Критическое условие:** Если целевой столбец (target_column) тоже является категориальным, трансформация OneHotEncoder не должна к нему применяться. Удаляет исходные текстовые категориальные столбцы, заменяя их бинарными.

In [104]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_column):
        # В конструкторе мы ТОЛЬКО запоминаем имя таргета. Данные сюда не подаем!
        self.target_column = target_column
        # sparse_output=False возвращает массив NumPy вместо разреженной матрицы SciPy (не хранит нули, а записывает в память только координаты единиц (например: "в строке 5, столбце 2 стоит единица").), которую пандас не переварит.
        # handle_unknown="ignore" защищает от падения (KeyError), если в валидационной
        # или тестовой выборке появятся новые категории, которых не было при обучении.
        self.encoder = OneHotEncoder(
            sparse=False, handle_unknown="ignore"
        )  # создаем экземпляр класса OneHotEncoder со всеми нужными настройками и сохраняем его в атрибут self.encoder
        self.categorical_cols_ = []

    def fit(self, X, y=None):
        # Сюда датафрейм прилетает под именем X.
        # Отделяем целевую переменную, если она присутствует в матрице X
        X_features = (
            X.drop(columns=[self.target_column])
            if self.target_column in X.columns
            else X
        )

        # Находим и запоминаем категориальные признаки
        self.categorical_cols_ = X_features.select_dtypes(
            include=["object", "category"]
        ).columns.tolist()  # Почему встроенный .tolist() чуть лучше, чем list(). Быстродействие (C-level оптимизация): Метод .tolist() в Pandas написан на языке C под капотом. Он превращает индекс столбцов в Python-список мгновенно. Функция list() заставляет Python вручную перебирать элементы в цикле, что работает медленнее на больших таблицах.

        # Обучаем внутренний OneHotEncoder только на признаках (без таргета!)
        if self.categorical_cols_:
            self.encoder.fit(X_features[self.categorical_cols_])
        return self

    def transform(self, X) -> Tuple[pd.DataFrame, pd.Series]:
        X_copy = X.copy()

        # Извлекаем и сохраняем целевую переменную (Series) отдельно, если она есть
        # y_series = (
        # X_copy[self.target_column] if self.target_column in X_copy.columns else pd.Series(dtype='object'))

        # вариант круче:
        y_series = X_copy.get(self.target_column, pd.Series(dtype="object"))

        X_features = (
            X_copy.drop(columns=[self.target_column])
            if self.target_column in X_copy.columns
            else X_copy
        )

        if self.categorical_cols_:
            # Трансформируем текстовые признаки в массивы чисел
            encoded_array = self.encoder.transform(X_features[self.categorical_cols_])

            # Получаем понятные названия для новых бинарных колонок
            encoded_cols = self.encoder.get_feature_names(self.categorical_cols_)
            encoded_df = pd.DataFrame(
                encoded_array, columns=encoded_cols, index=X_features.index
            )

            # Убираем старые текстовые признаки и приклеиваем закодированные бинарные ячейки
            X_features = X_features.drop(columns=self.categorical_cols_)
            X_features = pd.concat([X_features, encoded_df], axis=1)

        return X_features, y_series

3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).

In [105]:
class TrainValidationTest:
    def __init__(self, test_size=0.2, random_state=21):
        self.test_size = test_size
        self.random_state = random_state

    def split(self, X, y):
        if y.empty:  # добавила, так у меня реализован возврат пустой серии инкодером, если запрашиваемого столбца в датафрейме нет, как раз напоролась на эту ошибку, когда назвала таргет weekday, а по тз ниже идет запрос "dayofweek"
            raise ValueError(
                """Разделение выборки не произведено, таргет 'y' пуст.
                Проверьте, совпадает ли имя target_column в MyOneHotEncoder
                "с реальным именем столбца в датафрейме."""
            )
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state, stratify=y
        )

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_val,
            y_train_val,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y_train_val,
        )

        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [106]:
class ModelSelection:
    """- Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models"""

    def __init__(self, grids: list, grid_dict):
        self.grids = grids  # пример: grids = [gs_svm, gs_tree, gs_rf]
        self.grid_dict = (
            grid_dict  # пример: {0: "SVM", 1: "Decision Tree", 2: "Random Forest"}
        )

        # Нижнее подчеркивание в начале имени (_results_df) — это стандарт python, который используется для обозначения скрытых (private) атрибутов.
        self._results_df = None  # внутренний приватный датафрейм (появится позже)
        self.best_estimator_ = None  # выученная лучшая модель (появится после обучения)

    # Подчёркивание В КОНЦЕ (self.best_estimator_) — cтандарт sclearn для всех моделей и трансформеров: любой атрибут, который вычисляется или подбирается в процессе обучения (в методах fit или choose), обязан заканчиваться на одиночное нижнее подчёркивание.

    def choose(self, X_train, y_train, X_valid, y_valid):
        """ф-я для обучения и вывода прогресс-бара"""

        self.models_data = []

        for idx, gs_model in enumerate(self.grids):
            gs_model_name = self.grid_dict[idx]  # имя модели для лога
            print(f"Estimator: {gs_model_name}")

            ######## расчет количества шагов для визуализации в tqdm ########
            n_combinations = len(ParameterGrid(gs_model.param_grid))
            # cv_folds = gs_model.cv if gs_model.cv is not None else 5
            cv_folds = gs_model.cv or 5
            total_steps = n_combinations * cv_folds

            # подключаем встроенный текстовый лог фолдов внутри самого sklearn, обычно его передавала в гридсерче параметром
            gs_model.set_params(verbose=1)

            # внешний прогресс-бар на время обучения модели
            with tqdm(total=total_steps, desc="") as pbar:
                gs_model.fit(X_train, y_train)
                pbar.update(total_steps)  # заполняем до 100% по завершении

            best_params = gs_model.best_params_
            best_training_acc = gs_model.best_score_
            best_model = gs_model.best_estimator_
            y_pred_valid = best_model.predict(X_valid)
            validation_acc = accuracy_score(y_pred_valid, y_valid)

            self.models_data.append(
                {
                    "model": gs_model_name,
                    "params": gs_model.best_params_,
                    "valid_score": validation_acc,
                    "estimator": gs_model.best_estimator_,
                }
            )

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_training_acc}")
            print(f"Validation set accuracy score for best params:  {validation_acc}")

        best_model_data = max(self.models_data, key=lambda x: x["valid_score"])
        best_model_name = best_model_data["model"]

        self.best_estimator_ = best_model_data["estimator"]

        print(f"Classifier with best validation set accuracy: {best_model_name}")

    def best_results(self):
        self._results_df = pd.DataFrame(self.models_data)
        return self._results_df


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [107]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator
        self.test_score_ = None

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)

        y_pred = self.estimator.predict(X_test)
        self.test_score_ = accuracy_score(y_test, y_pred)
        print(f"Accuracy of the final model is {self.test_score_}")
        return self.test_score_

    def save_model(self, path):
        model_name = type(self.estimator).__name__
        file_name = f"{model_name}_{self.test_score_}.sav"
        full_path = os.path.join(path, file_name)
        joblib.dump(self.estimator, full_path)
        print(f"Model {model_name} was successfully saved to {full_path}")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [108]:
notebook_start_time = time.time()

1. Load the data from the file (****name of file****).

In [109]:
df = pd.read_csv("../data/checker_submits.csv")

2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])

In [110]:
# Pipeline is a sequence of data transformers with an optional final predictor.
preprocessing = Pipeline(
    [
        ("feature_extractor", FeatureExtractor()),
        ("onehot_encoder", MyOneHotEncoder(target_column="dayofweek")),
    ]
)

3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```

In [111]:
data = preprocessing.fit_transform(df)

4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.

In [112]:
X_features, y_series = data

X_train, X_valid, X_test, y_train, y_valid, y_test = TrainValidationTest().split(
    X_features, y_series
)

5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.

In [113]:
svm = SVC(random_state=21)
dt = DecisionTreeClassifier(random_state=21)
rf = RandomForestClassifier(random_state=21)


svm_params = [
    {
        "kernel": ("linear", "rbf", "sigmoid"),
        "C": [0.01, 0.1, 1, 1.5, 5, 10],
        "gamma": ["scale", "auto"],
        "class_weight": ("balanced", None),
        "random_state": [21],
        "probability": [True],
    }
]

dt_params = [
    {
        "max_depth": range(5, 30),
        "criterion": ("entropy", "gini"),
        "class_weight": ("balanced", None),
    }
]

rf_params = [
    {
        "n_estimators": (5, 10, 50, 100),
        "max_depth": range(1, 30),
        "criterion": ("entropy", "gini"),
        "class_weight": ("balanced", None),
    }
]

gs_svm = GridSearchCV(
    estimator=svm, param_grid=svm_params, scoring="accuracy", cv=2, n_jobs=-1
)

gs_tree = GridSearchCV(
    estimator=dt, param_grid=dt_params, scoring="accuracy", cv=2, n_jobs=-1
)

gs_rf = GridSearchCV(
    estimator=rf, param_grid=rf_params, scoring="accuracy", cv=2, n_jobs=-1
)

grids = [gs_svm, gs_tree, gs_rf]
grids_dict = {0: "SVM", 1: "Decision Tree", 2: "Random Forest"}

selector = ModelSelection(grids, grids_dict)
chosenModelname = selector.choose(X_train, y_train, X_valid, y_valid)
best_estimator = selector.best_estimator_
chosenDf = selector.best_results()

Estimator: SVM


  0%|          | 0/144 [00:00<?, ?it/s]

Fitting 2 folds for each of 72 candidates, totalling 144 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done 144 out of 144 | elapsed:   21.9s finished


Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.7727272727272727
Validation set accuracy score for best params:  0.8777777777777778
Estimator: Decision Tree


  0%|          | 0/200 [00:00<?, ?it/s]

... (6 outputs left out)

Fitting 2 folds for each of 464 candidates, totalling 928 fits


[Parallel(n_jobs=-1)]: Done 772 tasks      | elapsed:    3.2s


Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50}
Best training accuracy: 0.8552875695732838
Validation set accuracy score for best params:  0.9074074074074074
Classifier with best validation set accuracy: Random Forest


[Parallel(n_jobs=-1)]: Done 928 out of 928 | elapsed:    3.7s finished



6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

In [ ]:
fin_est = Finalize(best_estimator)
final_score = fin_est.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model is 0.9053254437869822


In [115]:
path = "../data"
fin_est.save_model(path)

Model RandomForestClassifier was successfully saved to ../data/RandomForestClassifier_0.9053254437869822.sav


In [116]:
notebook_total_time = time.time() - notebook_start_time

minutes = int(notebook_total_time // 60)
seconds = int(notebook_total_time % 60)

display(
    Markdown(
        f"""
---
**Время работы блокнота:** <span style="color:green">**{minutes} мин. {seconds} сек.**</span>

---
"""
    )
)


---
**Время работы блокнота:** <span style="color:green">**0 мин. 26 сек.**</span>

---


In [118]:
loaded_model = joblib.load("../data/RandomForestClassifier_0.9053254437869822.sav")
y_pred_loaded = loaded_model.predict(X_test)
loaded_accuracy = accuracy_score(y_test, y_pred_loaded)

print(
    f"Accuracy загруженной модели: {loaded_accuracy}\nAccuracy сохраненной модели: {final_score}"
)

Accuracy загруженной модели: 0.9053254437869822
Accuracy сохраненной модели: 0.9053254437869822
